#NFL Play by Play 2009–2017 (v4) — roteiro de pré-processamento


Roteiro de estudo: complete as células marcadas com TODO, execute cada etapa e escreva suas próprias justificativas. Este arquivo é um guia de trabalho, não um resultado final pronto para entrega.



#Objetivo
Diagnosticar, tipificar, tratar valores ausentes, limpar inconsistências, exportar para Parquet e validar a preservação dos dados.

#1. Preparação e carregamento

Informe o caminho do arquivo original. Registre também a versão do dataset utilizada e a origem do arquivo.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# TODO: informe o caminho do CSV original
arquivo_csv = Path("NFL_2009_2017.csv")

# TODO: carregue o arquivo e confira se o separador e a codificação estão corretos
# Como o arquivo pode ser grande e conter tipos mistos, low_memory=False evita avisos do pandas
df = pd.read_csv(arquivo_csv, low_memory=False)

# TODO: só prossiga após definir df
print("Dataset carregado com sucesso!")

Dataset carregado com sucesso!


#2. Diagnóstico inicial

Registre dimensões, nomes das colunas, tipos atuais, quantidade de valores ausentes e linhas duplicadas. Interprete os resultados em suas próprias palavras.

In [3]:
print(df.shape)
display(df.head())
display(df.dtypes)
df.info(memory_usage="deep")

ausentes = pd.DataFrame({
    "quantidade": df.isna().sum(),
    "percentual": df.isna().mean() * 100
})

display(ausentes.sort_values("percentual", ascending=False))
print("Linhas duplicadas:", df.duplicated().sum())

(407688, 102)


,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


Date            str
GameID        int64
Drive         int64
qtr           int64
down        float64
             ...   
Win_Prob    float64
WPA         float64
airWPA      float64
yacWPA      float64
Season        int64
Length: 102, dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Columns: 102 entries, Date to Season
dtypes: float64(33), int64(31), str(38)
memory usage: 380.5 MB


,quantidade,percentual
DefTwoPoint,407664,99.994113
BlockingPlayer,407571,99.971302
TwoPointConv,407083,99.851602
ChalReplayResult,404286,99.165538
RecFumbPlayer,403315,98.927366
...,...,...
HomeTeam,0,0.000000
Timeout_Indicator,0,0.000000
TwoPoint_Prob,0,0.000000
ExPoint_Prob,0,0.000000


Linhas duplicadas: 0


#3. Tipificação dos atributos

Use o significado das colunas para decidir entre tipos numéricos, booleanos, datas/tempos, texto e categorias. Não converta identificadores em números apenas porque contêm dígitos. Antes de converter, examine valores únicos e possíveis códigos especiais.

In [ ]:
# TODO: examine os tipos atuais e os valores distintos das colunas que pretende converter
# TODO: defina um plano de tipificação e explique as decisões em Markdown
# TODO: aplique conversões com errors="coerce" quando apropriado e quantifique conversões que viraram ausentes


In [ ]:
display(df.dtypes)
display(df.nunique().sort_values())

colunas_numericas = df.select_dtypes(include=["number"]).columns
colunas_texto = df.select_dtypes(include=["object", "string"]).columns

print("Colunas numéricas:", list(colunas_numericas))
print("Colunas textuais:", list(colunas_texto))

,0
Date,object
GameID,int64
Drive,int64
qtr,int64
down,float64
...,...
Win_Prob,float64
WPA,float64
airWPA,float64
yacWPA,float64


,0
PlayAttempted,1
FirstDown,2
Safety,2
GoalToGo,2
PuntResult,2
...,...
Away_WP_pre,362562
Home_WP_post,363190
Away_WP_post,363360
WPA,366549


Colunas numéricas: ['GameID', 'Drive', 'qtr', 'down', 'TimeUnder', 'TimeSecs', 'PlayTimeDiff', 'yrdln', 'yrdline100', 'ydstogo', 'ydsnet', 'GoalToGo', 'FirstDown', 'PlayAttempted', 'Yards.Gained', 'sp', 'Touchdown', 'Safety', 'Onsidekick', 'PassAttempt', 'AirYards', 'YardsAfterCatch', 'QBHit', 'InterceptionThrown', 'RushAttempt', 'Reception', 'FieldGoalDistance', 'Fumble', 'Sack', 'Challenge.Replay', 'Accepted.Penalty', 'Penalty.Yards', 'PosTeamScore', 'DefTeamScore', 'ScoreDiff', 'AbsScoreDiff', 'Timeout_Indicator', 'posteam_timeouts_pre', 'HomeTimeouts_Remaining_Pre', 'AwayTimeouts_Remaining_Pre', 'HomeTimeouts_Remaining_Post', 'AwayTimeouts_Remaining_Post', 'No_Score_Prob', 'Opp_Field_Goal_Prob', 'Opp_Safety_Prob', 'Opp_Touchdown_Prob', 'Field_Goal_Prob', 'Safety_Prob', 'Touchdown_Prob', 'ExPoint_Prob', 'TwoPoint_Prob', 'ExpPts', 'EPA', 'airEPA', 'yacEPA', 'Home_WP_pre', 'Away_WP_pre', 'Home_WP_post', 'Away_WP_post', 'Win_Prob', 'WPA', 'airWPA', 'yacWPA', 'Season']
Colunas textuais:

#4. Tratamento de valores ausentes
Não aplique uma única estratégia a todas as colunas. Diferencie ausência estrutural (por exemplo, um campo que não se aplica à jogada) de ausência que representa falha de registro. Compare a proporção de ausentes e justifique cada decisão. Evite imputar valores sem justificativa semântica.

In [ ]:
# TODO: crie uma tabela com coluna, ausentes, percentual e tipo
# TODO: selecione estratégias por coluna/grupo de colunas
# TODO: após o tratamento, compare a contagem de ausentes antes e depois


In [ ]:
ausentes_antes = df.isna().sum()

diagnostico_ausentes = pd.DataFrame({
    "quantidade": ausentes_antes,
    "percentual": df.isna().mean() * 100
})

display(diagnostico_ausentes.sort_values("percentual", ascending=False))

,quantidade,percentual
DefTwoPoint,407664,99.994113
BlockingPlayer,407571,99.971302
TwoPointConv,407083,99.851602
ChalReplayResult,404286,99.165538
RecFumbPlayer,403315,98.927366
...,...,...
HomeTeam,0,0.000000
Timeout_Indicator,0,0.000000
TwoPoint_Prob,0,0.000000
ExPoint_Prob,0,0.000000


#5. Limpeza e consistência
Verifique duplicatas exatas e, separadamente, se há uma chave natural plausível para identificar jogadas repetidas. Inspecione categorias, espaços extras, grafias divergentes, intervalos impossíveis e valores especiais. Não remova registros sem justificar o critério.

In [ ]:
# TODO: investigue duplicatas
# TODO: investigue categorias e strings inconsistentes
# TODO: defina e aplique regras de limpeza justificadas
# TODO: registre quantas linhas/valores foram alterados ou removidos


In [ ]:
print("Duplicatas:", df.duplicated().sum())

colunas_texto = df.select_dtypes(include=["object", "string"]).columns

for coluna in colunas_texto:
    print(f"\n{coluna}")
    print(df[coluna].value_counts(dropna=False).head(10))

Duplicatas: 0

Date
Date
2016-01-03    2872
2012-01-01    2825
2017-01-01    2819
2017-12-31    2801
2011-01-02    2772
2014-12-28    2771
2012-12-30    2737
2013-12-29    2729
2010-01-03    2716
2009-09-20    2674
Name: count, dtype: int64

time
time
15:00    11792
02:00     7012
00:00     6914
14:55     1174
01:55     1001
14:54      980
01:56      898
01:54      840
14:56      830
00:35      783
Name: count, dtype: int64

SideofField
SideofField
OAK    13270
CLE    13212
BUF    13094
TEN    12968
MIA    12924
NYG    12855
ARI    12834
DET    12799
TB     12788
IND    12760
Name: count, dtype: int64

posteam
posteam
NaN    24992
NE     12520
NO     12408
DEN    12398
PHI    12397
HOU    12315
DET    12271
BAL    12241
IND    12156
NYJ    12076
Name: count, dtype: int64

DefensiveTeam
DefensiveTeam
NaN    24992
PHI    12321
TEN    12293
BUF    12237
ARI    12236
CLE    12228
DEN    12222
OAK    12158
MIA    12133
SF     12109
Name: count, dtype: int64

desc
desc
Two-Minute Warning    

#6. Otimização de armazenamento
Considere tipos inteiros menores apenas depois de conferir os limites e a presença de ausentes. Considere category para colunas textuais repetitivas e string para texto livre. Compare a memória do DataFrame antes e depois sem sacrificar a informação.

In [ ]:
# TODO: registre memória antes/depois da tipificação
# TODO: avalie quais colunas podem usar tipos mais compactos com segurança
# TODO: documente o motivo de cada otimização


In [ ]:
memoria_bytes = df.memory_usage(deep=True).sum()
memoria_mb = memoria_bytes / (1024 ** 2)

print(f"Memória utilizada: {memoria_mb:.2f} MB")

Memória utilizada: 826.11 MB


#7. Exportação para Parquet
Escolha um caminho de saída e registre o tamanho do arquivo original e do Parquet. Se o CSV estiver compactado ou houver arquivos auxiliares, descreva exatamente o que está sendo comparado.

In [ ]:
# TODO: defina o caminho do Parquet
# parquet_path = Path("NFL_tratado.parquet")

# TODO: exporte sem salvar o índice
# df.to_parquet(parquet_path, index=False)

# TODO: calcule tamanhos em bytes/MB e a redução percentual


In [ ]:
parquet_path = Path("/content/NFL_tratado.parquet")

df.to_parquet(parquet_path, index=False)

tamanho_original = arquivo_csv.stat().st_size
tamanho_parquet = parquet_path.stat().st_size

print(f"CSV original: {tamanho_original / 1024**2:.2f} MB")
print(f"Parquet: {tamanho_parquet / 1024**2:.2f} MB")

reducao = (1 - tamanho_parquet / tamanho_original) * 100

print(f"Redução de tamanho: {reducao:.2f}%")

CSV original: 262.77 MB
Parquet: 73.00 MB
Redução de tamanho: 72.22%


#8. Validação após salvar
Leia o Parquet de volta e compare número de linhas, colunas, nomes, tipos, valores ausentes e conteúdo. Para comparação exata, considere as diferenças legítimas de representação entre tipos; explique qualquer diferença observada.

In [ ]:
# TODO: leia o Parquet salvo
# df_lido = pd.read_parquet(parquet_path)

# TODO: compare dimensões e colunas
# TODO: compare tipos e contagem de ausentes
# TODO: valide igualdade dos dados e investigue divergências


In [ ]:
df_validacao = pd.read_parquet(parquet_path)

print("Dimensões originais:", df.shape)
print("Dimensões após leitura:", df_validacao.shape)

print("Colunas iguais:", df.columns.equals(df_validacao.columns))

display(pd.DataFrame({
    "antes": df.isna().sum(),
    "depois": df_validacao.isna().sum()
}))

print("Tipos antes:")
print(df.dtypes)

print("Tipos depois:")
print(df_validacao.dtypes)

Dimensões originais: (407688, 102)
Dimensões após leitura: (407688, 102)
Colunas iguais: True


,antes,depois
Date,0,0
GameID,0,0
Drive,0,0
qtr,0,0
down,61154,61154
...,...,...
Win_Prob,25009,25009
WPA,5541,5541
airWPA,248501,248501
yacWPA,248762,248762


Tipos antes:
Date         object
GameID        int64
Drive         int64
qtr           int64
down        float64
             ...   
Win_Prob    float64
WPA         float64
airWPA      float64
yacWPA      float64
Season        int64
Length: 102, dtype: object
Tipos depois:
Date         object
GameID        int64
Drive         int64
qtr           int64
down        float64
             ...   
Win_Prob    float64
WPA         float64
airWPA      float64
yacWPA      float64
Season        int64
Length: 102, dtype: object


#9. Conclusão
Escreva um resumo próprio incluindo:
- problemas encontrados no diagnóstico;
- conversões de tipo realizadas e justificativas;
- estratégias aplicadas aos valores ausentes;
- duplicatas/inconsistências tratadas;
- tamanho original e tamanho Parquet, com percentual de redução;
- verificações de preservação e limitações restantes.